# Day 2: Streaming Ingestion (Kafka / MSK + Spark Streaming)

This notebook simulates streaming ingestion of synthetic FHIR Observation & Patient records into the Bronze Iceberg table using PySpark memory streams.

In [ ]:
!pip install pyspark pyiceberg duckdb splink pydeequ -q

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit, col, struct, to_json, expr
from pyspark.sql.types import StructType, StructField, StringType

# Set up Spark Session with Iceberg configurations
spark = SparkSession.builder \
    .appName("Day02_Streaming_Ingestion") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.3") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", "/tmp/iceberg_warehouse") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark session initialized with Iceberg catalog.")

In [ ]:
# Re-create the Bronze table if it doesn't exist (from Day 1)
create_table_sql = """
CREATE TABLE IF NOT EXISTS local.db.bronze_raw_payload (
    ingestion_timestamp TIMESTAMP,
    source_system_id STRING,
    payload_id STRING,
    raw_payload_json STRING
)
USING iceberg
PARTITIONED BY (days(ingestion_timestamp))
"""

spark.sql("CREATE DATABASE IF NOT EXISTS local.db")
spark.sql(create_table_sql)

print("Bronze Iceberg table ready.")

In [ ]:
# Simulate a streaming source using rate source
streaming_df = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 2) \
    .load()

# Construct synthetic FHIR JSON payloads
bronze_stream_df = streaming_df.select(
    current_timestamp().alias("ingestion_timestamp"),
    lit("MOCK_FHIR_STREAM").alias("source_system_id"),
    col("value").cast("string").alias("payload_id"),
    to_json(struct(
        lit("Patient").alias("resourceType"),
        col("value").cast("string").alias("id"),
        lit("active").alias("status")
    )).alias("raw_payload_json")
)

# Write stream to Iceberg table
query = bronze_stream_df.writeStream \
    .format("iceberg") \
    .outputMode("append") \
    .option("checkpointLocation", "/tmp/iceberg_checkpoints/fhir_stream") \
    .toTable("local.db.bronze_raw_payload")

import time
time.sleep(10) # Let it run for a bit to generate records
query.stop()
print("Streaming ingestion simulation complete.")

In [ ]:
# Verify data ingested into Bronze Iceberg
spark.sql("SELECT * FROM local.db.bronze_raw_payload ORDER BY ingestion_timestamp DESC LIMIT 10").show(truncate=False)